## Datamining using cotraining

This notebook contains a small visualization of the results of datamining using cotraining. 

It uses a Gaussian Mixture model to cluster the data, and add labels for a few datapoints, while leaving most data unlabeled. Three different classifiers (MLP, Random Forest and XGBoost) are then trained on this small labled dataset. Their best predictions are treated as true, and are added to the dataset. The idea is that the predictions become better with each iteration, thus allowing for higher confidence labels. Eventually, our new dataset should resemble the original one.

In the end, the results are visualized and compared to the original data. 

This notebook takes roughly 1 minute to run.

### Libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.mixture import GaussianMixture
from scipy.stats import entropy
from xgboost import XGBClassifier
from random import randint

%matplotlib inline

### Generate random datapoints

First we generate a number of random datapoints.

In [ ]:
n = 2000        # Number of total datapoints

x = np.random.random_sample((n))*randint(-1000, 1000) + randint(-350, 350)
y = np.random.random_sample((n))*randint(-1000, 1000) + randint(-350, 350)

data = np.column_stack((x, y))
data = np.unique(data, axis=0)

data.shape

This is what the generated data looks like

In [ ]:
fig, ax = plt.subplots(figsize= (10, 5), facecolor="lightgrey")
ax.scatter(x, y, s=10)
ax.set_title("Random data");

### Divide into clusters and label data

The Gaussian Mixture model divides the data into clusters and assigns our first labled data.

In [ ]:
n_clusters = 5
ratio = 0.05      # Percentage of intital labled samples

gmm = GaussianMixture(n_components= n_clusters)
gmm.fit(data)

labels = gmm.predict(data)      # Clustering

indices = np.random.choice([False, True], size=(data.shape[0],), p=[1-ratio, ratio])

labeled_data = data[indices]
remaining_data = data[~indices]

new_labels = gmm.predict(labeled_data)
original_labels = new_labels                # Initial labels for the labled data. Save for later.

print(f"There are {len(labeled_data)} labled datapoints and {len(remaining_data)} unlabled datapoints. That's {len(remaining_data)+len(labeled_data)} datapoints in total.")

Here's the class distribution of our initial samples. This is the data we'll be using for the cotraining.

In [ ]:
fig, ax = plt.subplots(figsize= (8, 5), facecolor="lightgrey")
values, counts = np.unique(original_labels, return_counts=True)

ax.bar(values, counts, color=plt.cm.tab20(np.arange(len(values))));
ax.set_title("Class distribution");

### Visualize clusters

In [ ]:
fig = plt.figure(figsize= (10, 10), facecolor="lightgrey")
axes = fig.subplots(nrows=2, ncols=1)

axes[0].scatter(x, y, s=10, alpha=0.8, label= "Unlabled data", edgecolors="black", linewidths=0.3)
axes[0].set_title("Random data")
axes[0].scatter(labeled_data[:,0], labeled_data[:,1], s=35, color="red", edgecolors="black", linewidths=0.85, label="Labeled data")
axes[0].legend()

axes[1].set_title("Clusters")
axes[1].scatter(data[:, 0], data[:, 1], c=labels, s=15, cmap='tab10', edgecolors="black", linewidths=0.3);

### Cotraining main loop

This is the most exciting part:

1. Both models are trained on the labled data.
2. Both make predictions on the remaining data.
3. The highest confidence predictions are treated as true and are added to the labled data.
4. Repeat until all samples are labled, no more progress can be made.


In [ ]:
cls_1 = RandomForestClassifier()
cls_2 = MLPClassifier(hidden_layer_sizes=(32, 32), max_iter=10000)
cls_3 = XGBClassifier()

labeled_data = np.array(labeled_data)
new_labels = np.array(new_labels)

print(f"Started with {len(labeled_data)} labled samples.")

top = 100                               # Max nr of labels to give out each round
entropy_threshold = 2 / n_clusters      # Adjust if necessary
prob_threshold = 0.95

classifiers = (cls_1, cls_2, cls_3)
epochs = 1 + len(remaining_data) // top

for _ in range(epochs):
    for cls in classifiers:
        cls.fit(labeled_data, new_labels)

    best_predictions = list()

    for cls in classifiers:
        predictions = cls.predict(remaining_data)
        prob = cls.predict_proba(remaining_data)

        max_probs = np.max(prob, axis=1)
        entropies = entropy(prob, axis=1)

        above_threshold = (entropies < entropy_threshold) & (max_probs > prob_threshold)
        indx = np.flatnonzero(above_threshold)

        if len(indx) > 0:
            best_predictions.append(pd.DataFrame({
                "index": indx,
                "prediction": predictions[indx],
                "prob": max_probs[indx],
                "entropy": entropies[indx],
            }))

    if not best_predictions:
        break

    best_predictions = pd.concat(best_predictions, ignore_index=True)

    best_predictions = (best_predictions
                    .sort_values(["prob", "entropy"], ascending=[False, True])
                    .drop_duplicates("index")
                    .head(top)                      # Only add limited amount of labels each epoch
                    )

    selected = best_predictions["index"].to_numpy()

    labeled_data = np.concatenate([labeled_data, remaining_data[selected]])

    new_labels = np.concatenate([new_labels, best_predictions["prediction"].to_numpy()])

    remaining_data = np.delete(remaining_data, selected, axis=0)

print(f"Finished with {len(labeled_data)} labled samples.")

### Results

Now we compare the labled data to the original clusters.

In [ ]:
fig = plt.figure(figsize= (10, 10), facecolor="lightgrey")
axes = fig.subplots(nrows=3, ncols=1)

labeled_data = np.array(labeled_data)

axes[0].set_title("Labled data")
axes[0].scatter(labeled_data[:, 0], labeled_data[:, 1], c=new_labels, cmap='viridis', s=13, edgecolors="black", linewidths=0.3);

axes[1].set_title("Original clusters");
axes[1].scatter(data[:, 0], data[:, 1], c=labels, s=15, cmap='viridis', edgecolors="black", linewidths=0.3);

if len(remaining_data) != 0:
    axes[2].set_title("Remaining data")
    axes[2].scatter(remaining_data[:, 0], remaining_data[:, 1], s=10, alpha=0.8, label= "Unlabled data", edgecolors="black", linewidths=0.3)
else:
    fig.delaxes(axes[2])

fig.tight_layout()

Now we compare the distribution of the initial samples, the new samples, and the original clusters.

In [ ]:
fig, axes = plt.subplots(figsize= (8, 8), nrows=2, ncols=1, facecolor="lightgrey")

values, counts = np.unique(new_labels, return_counts=True)
axes[0].bar(values, counts, color=plt.cm.tab20(np.arange(len(values))), zorder=3);
axes[0].set_title("Distribution of the new samples")

values, counts = np.unique(labels, return_counts=True)
axes[1].bar(values, counts, color=plt.cm.tab20(np.arange(len(values))), zorder=3);
axes[1].set_title("Distribution of the original clusters")


for ax in axes:
    ax.set_ylabel("Frequency")
    ax.set_xlabel("Cluster")
    ax.grid(axis="y", zorder=0)

fig.tight_layout()

The new samples should have a similar class distribution to the original clusters.

#### Finally, we evaluate our new dataset by comparing it to the original data.

In [ ]:
original = {tuple(x): y for x, y in zip(data, labels)}

predicted = {tuple(x): y for x, y in zip(labeled_data, new_labels)}

found = np.array([tuple(x) in predicted for x in data])

correct = np.array([
    found[i] and predicted[tuple(x)] == labels[i]
    for i, x in enumerate(data)
])

found_count = found.sum()
correct_count = correct.sum()

accuracy = correct_count / found_count if found_count else 0.0
accuracy_all = correct_count / len(data)
coverage = found_count / len(data)

print(f"Labeled accuracy:\t{accuracy:.3%}")
print(f"Overall accuracy:\t{accuracy_all:.3%}")
print(f"Coverage:\t\t{coverage:.3%} ({found_count}/{len(data)})")
print(f"Correct:\t\t{correct_count}/{found_count}")
print(f"Incorrect:\t\t{found_count - correct_count}/{found_count}")